# 09 Temporal graph models (Family F)
Implemented because their temporal mechanism is distinct and testable on our snapshot stream:
- **TGAT-style**: attention with Time2Vec encodings of edge recency;
- **EvolveGCN-O**: GCN weights evolved by a GRU along the snapshot sequence;
- **PRESAGE graph encoder**: HGT over recency-split relations (`pays_recent`/`pays_old`) with a per-account GRU memory across snapshots.

Not implemented, on purpose: TGN/DyRep/JODIE/GraphMixer need continuous-time event batching and memory modules whose incremental value over TGAT + snapshot memory could not be separated on a 120-day synthetic stream; adding them would inflate the model count, not the insight. They are listed as next experiments.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
g = table('graph_models.csv'); g[g.model.isin(['none','gcn','sage','hgt','tgat','evolvegcn'])][['model','pr_auc','pr_auc_lo','pr_auc_hi','recall','precision','f2','train_s']].round(3)

,model,pr_auc,pr_auc_lo,pr_auc_hi,recall,precision,f2,train_s
0,none,0.874,0.821,0.905,0.813,0.743,0.798,4.476
1,gcn,0.921,0.883,0.948,0.826,0.905,0.840,41.786
2,sage,0.924,0.875,0.950,0.829,0.915,0.845,21.963
5,hgt,0.959,0.921,0.976,0.900,0.936,0.907,383.606
7,tgat,0.942,0.904,0.963,0.910,0.925,0.913,101.790
8,evolvegcn,0.872,0.828,0.902,0.765,0.856,0.781,46.334


In [3]:
from presage.models.gnn import Time2Vec
import torch
t2v = Time2Vec(8); x = torch.linspace(0, 6, 5); print(t2v(x).shape, 'time encoding of recency (log-days) -> 8 dims')

torch.Size([5, 8]) time encoding of recency (log-days) -> 8 dims


RQ2/3 in this study: do temporal encoders beat static ones? Check the table above and the PRESAGE `no_temporal` ablation (notebook 14) before drawing a conclusion; differences between encoders are often inside the confidence intervals.